In [2]:
"""
============================================================
A2A PLANNER CLIENT
============================================================

This client demonstrates the A2A:

    input-required
    +
    task continuation

Example conversation:

    User:
        plan a trip to Kyoto

    Planner:
        How many days is the trip?

    User:
        3 days

    Planner:
        Day 1: explore Kyoto
        Day 2: explore Kyoto
        Day 3: explore Kyoto


============================================================
A2A WORKFLOW
============================================================

FIRST REQUEST
-------------

Client
   |
   | message/send
   | "plan a trip to Kyoto"
   v
Planner Agent
   |
   | execute()
   |
   | missing duration
   v
Task
   |
   | input-required
   v
Client
   |
   | "How many days?"
   v


SECOND REQUEST
--------------

Client
   |
   | message/send
   | "3 days"
   |
   | SAME task/context
   v
Planner Agent
   |
   | execute() again
   |
   | task.history
   |
   | "plan a trip to Kyoto"
   | "3 days"
   v
Task
   |
   | working
   v
Generate itinerary
   |
   | artifact
   v
Task
   |
   | completed
   v
Client


============================================================
IMPORTANT
============================================================

The second message belongs to the SAME task.

We therefore need to remember:

    task_id
    context_id

from the first response.

"""

import uuid

import httpx

from google.protobuf.json_format import MessageToJson

from a2a.client import (
    A2ACardResolver,
    ClientConfig,
    create_client,
)

from a2a.types import (
    Message,
    Part,
    Role,
    SendMessageRequest,
)


# ============================================================
# PLANNER AGENT URL
# ============================================================

AGENT_URL = "http://localhost:9103"


# ============================================================
# PROTOBUF PRINT HELPER
# ============================================================
#
# A2A SDK 1.x uses protobuf messages.
#
# Therefore we use:
#
#     MessageToJson()
#
# instead of:
#
#     model_dump()
#
# ============================================================

def print_proto(title, proto_msg):

    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)

    print(
        MessageToJson(
            proto_msg,
            indent=2,
        )
    )


# ============================================================
# CREATE A2A MESSAGE REQUEST
# ============================================================
#
# This helper keeps our main workflow clean.
#
# Every message gets:
#
#     message_id
#     role
#     text
#
# ============================================================

def create_request(text):

    return SendMessageRequest(

        message=Message(

            # Unique ID for this message.
            message_id=str(
                uuid.uuid4()
            ),

            # Message is from the user.
            role=Role.ROLE_USER,

            # Message content.
            parts=[
                Part(
                    text=text
                )
            ],
        )
    )


# ============================================================
# MAIN
# ============================================================

async def main():

    print("=" * 60)
    print("A2A PLANNER CLIENT")
    print("=" * 60)


    # ========================================================
    # STEP 1
    # CREATE HTTP CLIENT
    # ========================================================

    async with httpx.AsyncClient() as httpx_client:


        # ====================================================
        # STEP 2
        # DISCOVER PLANNER AGENT
        # ====================================================
        #
        # The resolver calls:
        #
        # GET
        #
        # http://localhost:9103/
        # .well-known/agent-card.json
        #
        # ====================================================

        print("\nGetting Planner Agent Card...")

        resolver = A2ACardResolver(
            httpx_client,
            AGENT_URL,
        )

        agent_card = await resolver.get_agent_card()


        print(
            "\nAgent:",
            agent_card.name,
        )

        print_proto(
            "PLANNER AGENT CARD",
            agent_card,
        )


        # ====================================================
        # STEP 3
        # CREATE A2A CLIENT
        # ====================================================
        #
        # Planner supports streaming.
        #
        # We therefore enable streaming on the client.
        #
        # ====================================================

        config = ClientConfig(
            httpx_client=httpx_client,
            streaming=True,
        )

        client = await create_client(
            agent_card,
            client_config=config,
        )


        # ====================================================
        # STEP 4
        # FIRST REQUEST
        # ====================================================
        #
        # User says:
        #
        #     "plan a trip to Kyoto"
        #
        # We expect:
        #
        #     Task
        #        ↓
        #     input-required
        #
        # because the user didn't specify duration.
        #
        # ====================================================

        first_text = "plan a trip to Kyoto"

        request = create_request(
            first_text
        )


        print("\n")
        print("=" * 60)
        print("FIRST REQUEST")
        print("=" * 60)

        print(
            "User:",
            first_text,
        )


        # ====================================================
        # STEP 5
        # SEND FIRST MESSAGE
        # ====================================================
        #
        # A2A operation:
        #
        #     message/send
        #
        # send_message() returns an async iterator.
        #
        # ====================================================

        task_id = None
        context_id = None

        requires_input = False


        async for event in client.send_message(
            request
        ):

            # ------------------------------------------------
            # Determine event type.
            # ------------------------------------------------

            kind = event.WhichOneof(
                "payload"
            )


            # =================================================
            # TASK EVENT
            # =================================================

            if kind == "task":

                task = event.task

                print(
                    "\nTask created:"
                )

                print(
                    "Task ID:",
                    task.id,
                )

                print(
                    "Context ID:",
                    task.context_id,
                )


                # ------------------------------------------------
                # IMPORTANT
                # ------------------------------------------------
                #
                # Save these IDs.
                #
                # We need them when continuing the task.
                #
                # ------------------------------------------------

                task_id = task.id

                context_id = task.context_id


            # =================================================
            # MESSAGE EVENT
            # =================================================

            elif kind == "message":

                texts = [
                    part.text
                    for part in event.message.parts
                    if part.text
                ]

                if texts:

                    print(
                        "\nPlanner:",
                        " ".join(texts),
                    )


            # =================================================
            # STATUS UPDATE
            # =================================================

            elif kind == "status_update":

                print(
                    "\nStatus update:"
                )

                print(
                    event.status_update
                )


                # ------------------------------------------------
                # Check whether the task requires user input.
                #
                # We use the protobuf representation here.
                # ------------------------------------------------

                status_update = (
                    event.status_update
                )

                print(
                    "Task is waiting for input."
                )

                requires_input = True


            # =================================================
            # ARTIFACT UPDATE
            # =================================================

            elif kind == "artifact_update":

                print(
                    "\nArtifact update:"
                )

                print(
                    event.artifact_update
                )


            # ------------------------------------------------
            # Print raw event while learning A2A.
            # ------------------------------------------------

            print_proto(
                f"RAW RESPONSE ({kind})",
                event,
            )


        # ====================================================
        # STEP 6
        # CHECK WHETHER WE NEED TO CONTINUE
        # ====================================================

        if not requires_input:

            print(
                "\nPlanner did not request additional input."
            )

            return


        # ====================================================
        # STEP 7
        # CONTINUE THE SAME TASK
        # ====================================================
        #
        # The user now answers:
        #
        #     "3 days"
        #
        # IMPORTANT:
        #
        # This is NOT a brand-new conversation.
        #
        # We continue the existing A2A task.
        #
        # We therefore reuse:
        #
        #     task_id
        #     context_id
        #
        # ====================================================

        second_text = "3 days"


        print("\n")
        print("=" * 60)
        print("CONTINUING EXISTING TASK")
        print("=" * 60)

        print(
            "Task ID:",
            task_id,
        )

        print(
            "Context ID:",
            context_id,
        )

        print(
            "User:",
            second_text,
        )


        # ====================================================
        # STEP 8
        # BUILD CONTINUATION REQUEST
        # ====================================================
        #
        # The message itself is:
        #
        #     "3 days"
        #
        # and we associate it with the existing task/context.
        #
        # ====================================================

        continuation_request = SendMessageRequest(

            message=Message(

                message_id=str(
                    uuid.uuid4()
                ),

                role=Role.ROLE_USER,

                parts=[
                    Part(
                        text=second_text
                    )
                ],

                # ------------------------------------------------
                # Continue the existing A2A context.
                # ------------------------------------------------

                context_id=context_id,

                # ------------------------------------------------
                # Continue the existing task.
                # ------------------------------------------------

                task_id=task_id,
            )
        )


        # ====================================================
        # STEP 9
        # SEND CONTINUATION
        # ====================================================
        #
        # Again:
        #
        #     message/send
        #
        # But this time the request belongs to the existing
        # task/context.
        #
        # ====================================================

        async for event in client.send_message(
            continuation_request
        ):

            kind = event.WhichOneof(
                "payload"
            )


            # =================================================
            # TASK
            # =================================================

            if kind == "task":

                print(
                    "\nTask:"
                )

                print(
                    event.task
                )


            # =================================================
            # MESSAGE
            # =================================================

            elif kind == "message":

                texts = [
                    part.text
                    for part in event.message.parts
                    if part.text
                ]

                if texts:

                    print(
                        "\nPlanner:",
                        " ".join(texts),
                    )


            # =================================================
            # STATUS UPDATE
            # =================================================

            elif kind == "status_update":

                print(
                    "\nStatus update:"
                )

                print(
                    event.status_update
                )


            # =================================================
            # ARTIFACT
            # =================================================
            #
            # The Planner Agent creates:
            #
            #     itinerary
            #
            # containing:
            #
            #     Day 1: explore Kyoto
            #     Day 2: explore Kyoto
            #     Day 3: explore Kyoto
            #
            # =================================================

            elif kind == "artifact_update":

                print(
                    "\nITINERARY:"
                )

                print(
                    event.artifact_update
                )


            # ------------------------------------------------
            # Raw event for learning/debugging.
            # ------------------------------------------------

            print_proto(
                f"RAW RESPONSE ({kind})",
                event,
            )


# ============================================================
# JUPYTER ENTRY POINT
# ============================================================
#
# Jupyter already has an asyncio event loop.
#
# Therefore:
#
#     await main()
#
# NOT:
#
#     asyncio.run(main())
#
# ============================================================

await main()

A2A PLANNER CLIENT

Getting Planner Agent Card...

Agent: planner

PLANNER AGENT CARD
{
  "name": "planner",
  "description": "Plans a trip and asks for duration if it was not supplied.",
  "supportedInterfaces": [
    {
      "url": "http://127.0.0.1:9103/",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ],
  "version": "1.0.0",
  "capabilities": {
    "streaming": true
  },
  "defaultInputModes": [
    "text/plain"
  ],
  "defaultOutputModes": [
    "text/plain"
  ],
  "skills": [
    {
      "id": "plan-trip",
      "name": "Plan a trip",
      "description": "Produces a day-by-day outline for a destination.",
      "tags": [
        "a2a",
        "learning-example"
      ],
      "examples": [
        "plan a trip to Kyoto",
        "plan 3 days in Lisbon"
      ],
      "inputModes": [
        "text/plain"
      ],
      "outputModes": [
        "text/plain"
      ]
    }
  ]
}


FIRST REQUEST
User: plan a trip to Kyoto

Task created:
Task ID: 449259d5